# T7-bonus · Telemetry as code

## Goal

Commit the App Insights + workspace Bicep, the environment export-toggle
Terraform, and the KQL dashboards/alert rules from `23` to the repo — so
observability isn't something one person clicked once.


## Prereqs

Asserted below, not just stated — this cell fails loudly if a prior notebook's step wasn't actually completed.


In [ ]:
from pathlib import Path
assert Path("../infra/bicep/modules/appinsights.bicep").exists()


## Concept

`23` proved the KQL queries work interactively. This notebook is the
"someone else can stand this up unattended" pass: the App Insights resource
from Bicep (already in the repo), the export toggle as a Terraform resource
alongside the rest of the platform layer, and the KQL itself saved as
versioned dashboard/alert definitions rather than living only in a
notebook cell someone has to remember to re-paste.


## Build


In [ ]:
from pathlib import Path
alerts_dir = Path("../infra/bicep/modules")
kql_dir = Path("../infra/kql")
kql_dir.mkdir(exist_ok=True)

(kql_dir / "tool-latency-by-agent.kql").write_text('''
AgentTraces
| where TimeGenerated > ago(1h)
| summarize p50=percentile(DurationMs, 50), p95=percentile(DurationMs, 95), count() by ToolName, AgentName
| order by p95 desc
''')

(kql_dir / "credit-attribution-by-agent.kql").write_text('''
AgentTraces
| where TimeGenerated > ago(24h)
| summarize TotalCredits=sum(CreditsConsumed) by AgentName
| order by TotalCredits desc
''')
print("KQL committed to infra/kql/ — same queries 23 ran interactively")


In [ ]:
# Alert rule: page if p95 tool latency exceeds 5s sustained for 10 minutes,
# or credit burn rate implies budget exhaustion within 24h.
alert_bicep = '''
resource highLatencyAlert 'Microsoft.Insights/scheduledQueryRules@2023-03-15-preview' = {
  name: '${namePrefix}-high-tool-latency'
  location: location
  properties: {
    severity: 2
    evaluationFrequency: 'PT5M'
    windowSize: 'PT10M'
    criteria: {
      allOf: [{
        query: loadTextContent('../kql/tool-latency-by-agent.kql')
        threshold: 5000
        operator: 'GreaterThan'
      }]
    }
  }
}
'''
(alerts_dir / "alerts.bicep").write_text(alert_bicep)
print("alert rule written")


## Verify

Same harness, same golden set, every notebook.


In [ ]:
import subprocess
result = subprocess.run(["az", "bicep", "build", "--file", "../infra/bicep/modules/alerts.bicep"], capture_output=True, text=True)
print(result.returncode, result.stderr[-500:] if result.returncode else "bicep compiles cleanly")


## Cost


In [ ]:
print("No agent build/publish — this notebook only commits telemetry IaC.")


## Teardown


In [ ]:
print("No teardown — telemetry-as-code is now the standing pattern for any new environment.")
